# Holter 모델 prediction 재평가

`BUNDLE`을 Holter handoff bundle 경로로 바꾼 뒤 실행합니다. 통계 특징·원시 파형 입력은 manifest의 선택 모델에 따라 평가되며 threshold는 validation Youden J에서 정합니다.

In [ ]:
import json
import re
import sys
from pathlib import Path

repo = Path.cwd()
sys.path.insert(0, str(repo / "Members/sang"))
from model.evaluation.evaluate_predictions import evaluate_files  # noqa: E402

BUNDLE = Path("Members/sang/model/outputs/handoff/REPLACE_WITH_RUN_LABEL")
manifest = json.loads((BUNDLE / "manifest.json").read_text(encoding="utf-8"))
model_name = manifest["selection"]["model"]
model_slug = re.sub(r"[^A-Za-z0-9._-]+", "_", model_name).strip("._-")
run_dir = BUNDLE / "runs" / model_slug
run_metadata = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
result = evaluate_files(
    run_dir / "validation_predictions.csv",
    test_path=(run_dir / "test_predictions.csv")
    if (run_dir / "test_predictions.csv").exists()
    else None,
    input_group=run_metadata["input_group"],
)
result